In [ ]:
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from scipy.stats import binned_statistic

filename = "tree_outfile_complete_tree_overlays.root"

file = uproot.open(filename)
tree = file["outree"]

print(file.keys())
print(tree)
print(tree.num_entries)
print(tree.keys())

plane = tree["plane"].array(library="np")
hit_wires = tree["hit_wires"].array(library="np")
tot_wires = tree["tot_wires"].array(library="np")
avg_pitch = tree["avg_pitch"].array(library="np")
g4id = tree["G4ID"].array(library="np")
truth_fraction = tree["truth_fraction"].array(library="np")
max_buco = tree["max_buco"].array(library="np")

In [ ]:
def TProfile(mask) :
    average_pitch = avg_pitch[mask]
    eff = hit_wires[mask]/tot_wires[mask]

    mean_y, x_edges, binnumber = binned_statistic(average_pitch, eff, statistic='mean', bins=400, range=(0,4))
    std_y, _, _ = binned_statistic(average_pitch, eff, statistic='std', bins=400, range=(0,4))

    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])

    counts, _ = np.histogram(average_pitch, bins=400, range=(0,4))
    err_y = std_y / np.sqrt(counts)

    return x_centers, mean_y, err_y

In [26]:
h = plt.hist2d(truth_fraction[max_buco < 10],g4id[max_buco < 10],bins=(100,5),range=[(0,1),(-1,4)],cmap='viridis', norm=LogNorm())
plt.xlabel('truth fraction', fontsize=18)
plt.ylabel('G4ID', fontsize=18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
cbar = plt.colorbar(h[3])

In [25]:
mask = (plane==2) & (g4id != -1) & (truth_fraction >= 0.8) & (max_buco < 10)
#mask = (plane==2) & (g4id == -1) 
h2_coll = plt.hist2d(avg_pitch[mask],hit_wires[mask]/tot_wires[mask],bins=(400,100),range=[(0,4),(0,1.0)],cmap='viridis', norm=LogNorm())
plt.xlabel('average pitch', fontsize=18)
plt.ylabel('efficiency', fontsize=18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
cbar = plt.colorbar(h2_coll[3])
plt.ylim(0.8,1)


In [27]:
x_centers_MC, mean_y_MC, err_y_MC = TProfile(((plane==2) & (g4id != -1) & (truth_fraction >= 0.8) & (max_buco < 10)))

plt.errorbar(x_centers_MC, mean_y_MC, yerr=err_y_MC, fmt='.')
plt.xlabel('average pitch', fontsize=18)
plt.ylabel('average efficiency', fontsize=18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlim(0,2)
plt.ylim(0.98,1.)